# AMTB Dose-Response Analysis: Lipid Droplet Measurements

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
import statsmodels.formula.api as smf

In [2]:
rng = np.random.default_rng(42)   # used by the bootstrap CIs
 
dosage_order = ['Control', '0.5uM', '1uM', '3uM', '5uM', '10uM']
params = ['Area', 'Perimeter', 'IntDen']

# 1. Load and clean

In [3]:
df = pd.read_csv('Dosage_Combined.csv')
df.columns = df.columns.str.strip().str.replace(' ', '_')
 
# RawIntDen is IntDen multiplied by a constant calibration factor (~93.2), so it holds no extra information. Drop it.
df = df.drop(columns=['RawIntDen'])
 
df['Dosage'] = pd.Categorical(df['Dosage'], categories=dosage_order, ordered=True)
 
# ImageID restarts at "snap 0" inside every dose, so the same name shows up in more than one group. Build a key that is unique per image.
df['image'] = df['Dosage'].astype(str) + '__' + df['ImageID'].astype(str)
 
print(f"droplets: {len(df)}")
print(f"images:   {df['image'].nunique()}")
print("\nimages per dose:")
print(df.groupby('Dosage', observed=True)['image'].nunique().to_string())
print("\ndroplets per image (min / median / max):")
sizes = df.groupby('image').size()
print(f"  {sizes.min()} / {int(sizes.median())} / {sizes.max()}")

droplets: 4878
images:   63

images per dose:
Dosage
Control    11
0.5uM      10
1uM        10
3uM        12
5uM        10
10uM       10

droplets per image (min / median / max):
  12 / 64 / 338


# 2. How clustered is the data? (shows why the fix is needed)

- Share of total variance that sits between images rather than within them
- Well above 0 means droplets in the same image resemble each other, i.e. they are not independent

In [4]:
print("\nbetween-image share of variance:")
for p in params:
    grand = df[p].mean()
    g = df.groupby('image')[p]
    ss_between = (g.size() * (g.mean() - grand) ** 2).sum()
    ss_within = g.apply(lambda x: ((x - x.mean()) ** 2).sum()).sum()
    print(f"  {p:10s}: {100 * ss_between / (ss_between + ss_within):.0f}%")


between-image share of variance:
  Area      : 11%
  Perimeter : 12%
  IntDen    : 12%


# 3. Collapse to one value per image (the main fix)

In [5]:
# Median per image, because the droplet distributions are very right-skewed.
img = (df.groupby(['Dosage', 'image'], observed=True)[params]
         .median()
         .reset_index())
img['Dosage'] = pd.Categorical(img['Dosage'], categories=dosage_order, ordered=True)
 
print("\nimage-level medians (mean +/- SD across images):")
print(img.groupby('Dosage', observed=True)[params].agg(['mean', 'std']).round(2).to_string())


image-level medians (mean +/- SD across images):
         Area       Perimeter         IntDen         
         mean   std      mean   std     mean      std
Dosage                                               
Control  1.48  0.34      5.13  0.55  2059.47   772.56
0.5uM    1.38  0.29      5.17  0.62  2334.30   578.62
1uM      1.34  0.45      5.12  0.88  2271.06   958.61
3uM      1.40  0.37      5.25  0.58  2422.94   802.85
5uM      1.31  0.55      4.90  1.12  2311.47  1485.65
10uM     1.01  0.31      4.41  0.71  1799.87  1066.34


# 4. Kruskal-Wallis on the image-level values

In [6]:
def epsilon_squared(H, n, k):
    # effect size for Kruskal-Wallis (0 = nothing, 1 = groups fully separated)
    return (H - k + 1) / (n - k)
 
print("\nKruskal-Wallis (image level):")
kw_p = {}
for p in params:
    groups = [img.loc[img.Dosage == d, p] for d in dosage_order]
    H, pval = stats.kruskal(*groups)
    kw_p[p] = pval
    e2 = epsilon_squared(H, len(img), len(dosage_order))
    print(f"  {p:10s}: H={H:5.2f}  p={pval:.3f}  eps^2={e2:.2f}  "
          f"({'significant' if pval < 0.05 else 'ns'})")


Kruskal-Wallis (image level):
  Area      : H=10.20  p=0.070  eps^2=0.09  (ns)
  Perimeter : H= 9.87  p=0.079  eps^2=0.09  (ns)
  IntDen    : H= 5.71  p=0.336  eps^2=0.01  (ns)


# 5. Dunn post-hoc, Bonferroni (image level)
- strictly this is only needed where Kruskal-Wallis was significant
- it is shown for all three so the result can be compared directly with the old droplet-level heatmaps

In [7]:
dunn_results = {}
print("\nDunn post-hoc, Bonferroni (image level):")
for p in params:
    d = sp.posthoc_dunn(img, val_col=p, group_col='Dosage', p_adjust='bonferroni')
    dunn_results[p] = d
    pairs = [f"{a} vs {b} (p={d.loc[a, b]:.3f})"
             for i, a in enumerate(dosage_order)
             for b in dosage_order[i + 1:] if d.loc[a, b] < 0.05]
    print(f"  {p:10s}: {pairs if pairs else 'no pair below 0.05'}")


Dunn post-hoc, Bonferroni (image level):
  Area      : no pair below 0.05
  Perimeter : no pair below 0.05
  IntDen    : no pair below 0.05


# 6. Mixed-effects model (uses every droplet but respects the grouping)

- log(value) ~ dose, with a separate baseline (random intercept) for each image
- same idea as steps 3-4 but it keeps all the data, so it has more power to detect a real effect

In [8]:
print("\nmixed model  log(value) ~ dose, random intercept per image:")
term = "C(Dosage, Treatment('Control'))[T.10uM]"   # the 10uM vs Control coefficient
for p in params:
    d = df.copy()
    d['y'] = np.log(d[p])
    full = smf.mixedlm("y ~ C(Dosage, Treatment('Control'))", d, groups=d['image']).fit(reml=False)
    null = smf.mixedlm("y ~ 1", d, groups=d['image']).fit(reml=False)
 
    lr = 2 * (full.llf - null.llf)                      # likelihood-ratio test
    p_overall = stats.chi2.sf(lr, len(dosage_order) - 1)
 
    b = full.params[term]
    p10 = full.pvalues[term]
    print(f"  {p:10s}: overall p={p_overall:.3f} | "
          f"10uM vs Control = x{np.exp(b):.2f} ({(np.exp(b) - 1) * 100:+.0f}%), p={p10:.3f}")


mixed model  log(value) ~ dose, random intercept per image:
  Area      : overall p=0.030 | 10uM vs Control = x0.67 (-33%), p=0.001
  Perimeter : overall p=0.072 | 10uM vs Control = x0.84 (-16%), p=0.006
  IntDen    : overall p=0.153 | 10uM vs Control = x0.76 (-24%), p=0.110


# 7. Dose-response trend (is there a consistent direction?)

In [9]:
# Spearman rank correlation between dose order and the image-level value.
print("\nrank correlation with dose (image level):")
dose_rank = {d: i for i, d in enumerate(dosage_order)}
img['drank'] = img['Dosage'].map(dose_rank)
for p in params:
    rho, pval = stats.spearmanr(img['drank'], img[p])
    print(f"  {p:10s}: rho={rho:+.2f}  p={pval:.3f}")


rank correlation with dose (image level):
  Area      : rho=-0.35  p=0.005
  Perimeter : rho=-0.31  p=0.014
  IntDen    : rho=-0.17  p=0.182


# 8. Are the three measures independent of each other?

In [10]:
print("\ncorrelation between endpoints:")
print(df[params].corr().round(2).to_string())


correlation between endpoints:
           Area  Perimeter  IntDen
Area       1.00       0.93    0.96
Perimeter  0.93       1.00    0.86
IntDen     0.96       0.86    1.00


# 9. Figures

In [11]:
def boot_ci(values, n_boot=2000):
    values = np.asarray(values)
    boot_means = [rng.choice(values, len(values), replace=True).mean()
                  for _ in range(n_boot)]
    return np.percentile(boot_means, [2.5, 97.5])
 
# 9a. image-level mean with a 95% bootstrap CI (honest version of mean +/- SEM)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Image-level mean with 95% bootstrap CI', fontweight='bold')
for ax, p in zip(axes, params):
    means, lo_err, hi_err = [], [], []
    for i, dse in enumerate(dosage_order):
        vals = img.loc[img.Dosage == dse, p].values
        m = vals.mean()
        lo, hi = boot_ci(vals)
        means.append(m); lo_err.append(m - lo); hi_err.append(hi - m)
        ax.plot(np.full(len(vals), i), vals, 'o', color='0.75', markersize=3, zorder=1)
    ax.errorbar(range(len(dosage_order)), means, yerr=[lo_err, hi_err],
                fmt='o-', color='steelblue', capsize=4, zorder=2)
    ax.set_xticks(range(len(dosage_order)))
    ax.set_xticklabels(dosage_order, rotation=30)
    ax.set_title(p); ax.set_xlabel('AMTB dose'); ax.set_ylabel(f'image median {p}')
plt.tight_layout()
plt.savefig('image_level_trend.png', dpi=150, bbox_inches='tight')
plt.close()
 
# 9b. distributions with the image medians drawn on top, so the real number of replicates (~10 per dose) is visible instead of thousands of points
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Droplet distributions with image medians overlaid', fontweight='bold')
palette = sns.color_palette('muted', len(dosage_order))
for ax, p in zip(axes, params):
    sns.violinplot(x='Dosage', y=p, data=df, order=dosage_order, hue='Dosage',
                   palette=palette, legend=False, inner=None, cut=0, ax=ax)
    sns.stripplot(x='Dosage', y=p, data=img, order=dosage_order,
                  color='black', size=7, edgecolor='white', linewidth=0.5, ax=ax)
    ax.set_title(p); ax.set_xlabel('AMTB dose'); ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('distributions_with_image_medians.png', dpi=150, bbox_inches='tight')
plt.close()
 
# 9c. corrected Dunn heatmaps (image level)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Dunn post-hoc, image level (Bonferroni)\n* = p < 0.05", fontweight='bold')
for ax, p in zip(axes, params):
    d = dunn_results[p]
    mask = np.triu(np.ones_like(d, dtype=bool))
    annot = d.map(lambda v: f"{v:.2f}*" if v < 0.05 else f"{v:.2f}")
    sns.heatmap(d, mask=mask, annot=annot, fmt='', cmap='RdYlGn_r',
                vmin=0, vmax=1, linewidths=0.5, cbar=(ax is axes[-1]), ax=ax)
    ax.set_title(p); ax.tick_params(axis='x', rotation=40); ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('dunn_image_level.png', dpi=150, bbox_inches='tight')
plt.close()
 
print("\nsaved: image_level_trend.png, distributions_with_image_medians.png, dunn_image_level.png")


saved: image_level_trend.png, distributions_with_image_medians.png, dunn_image_level.png
